# SwinIR SEM 微调 — 正式训练

GPU: 2×T4 | Checkpoint: 10k iter | 目标: 70k iter

**策略**：
1. 先跑 200 iter 试运行（验证训练能启动、ETA 合理）
2. 确认无误后，跑完整训练
3. 训练完成后下载 output 中的模型和日志

## 0. 环境准备（与 check.ipynb 相同）

In [1]:
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0))
        print(f'  GPU {i}: {p.name} — {mem/1e9:.1f} GB')

PyTorch: 2.10.0+cu128, CUDA: True, GPUs: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


In [2]:
import os, shutil
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

# 克隆仓库
if not os.path.exists('BasicSR'):
    !git clone https://github.com/Log-Dog012/BasicSR.git
    !cd BasicSR && git checkout cuda-sem-finetune

# 安装依赖
os.chdir('BasicSR')
!pip install -r requirements.txt -q
!pip install -e . -q
!pip install lpips timm -q
print('✅ 环境就绪')

Cloning into 'BasicSR'...
remote: Enumerating objects: 4804, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 4804 (delta 65), reused 68 (delta 43), pack-reused 4689 (from 3)
Receiving objects: 100% (4804/4804), 4.07 MiB | 15.59 MiB/s, done.
Resolving deltas: 100% (3181/3181), done.
Already on 'cuda-sem-finetune'
Your branch is up to date with 'origin/cuda-sem-finetune'.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 64.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 20.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.9 MB/s eta 0:00:00
✅ 环境就绪


In [3]:
# 复制模型文件
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/1'
swinir_zoo = os.path.join(WORK_DIR, 'BasicSR', 'SwinIR', 'model_zoo')
exp_dir = os.path.join(WORK_DIR, 'BasicSR', 'experiments', 'finetune_SwinIR_SRx4_SEM')

# 预训练权重
os.makedirs(swinir_zoo, exist_ok=True)
for root, dirs, files in os.walk(MODEL_INPUT):
    for f in files:
        if 'classicalSR' in f and f.endswith('.pth'):
            shutil.copy2(os.path.join(root, f), os.path.join(swinir_zoo, f))
            print(f'✅ 预训练: {f}')
            break

# Checkpoint
for name in ['net_g_10000.pth', '10000.state']:
    for root, dirs, files in os.walk(MODEL_INPUT):
        if name in files:
            dst_dir = os.path.join(exp_dir, 'models' if 'pth' in name else 'training_states')
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(os.path.join(root, name), os.path.join(dst_dir, name))
            print(f'✅ {name}')
            break
print('✅ 模型文件就绪')

✅ 预训练: 001_classicalSR_DIV2K_s48w8_SwinIR-M_x4.pth
✅ net_g_10000.pth
✅ 10000.state
✅ 模型文件就绪


## 1. 试运行（200 iter，确认训练正常）

In [4]:
# 试运行 200 iter — DDP 模式
# --force_yml 临时覆盖配置，不修改原 YAML
print('开始试运行 (200 iter, DDP 2xT4)...')
print('='*50)
!torchrun --nproc_per_node=2 --master_port=4321 -m basicsr.train \
  -opt options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml \
  --launcher pytorch --auto_resume \
  --force_yml train:total_iter=10200 logger:print_freq=50 \
    logger:save_checkpoint_freq=50000 val:val_freq=50000

开始试运行 (200 iter, DDP 2xT4)...
W0709 16:26:50.249000 192 torch/distributed/run.py:852] 
W0709 16:26:50.249000 192 torch/distributed/run.py:852] *****************************************
W0709 16:26:50.249000 192 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0709 16:26:50.249000 192 torch/distributed/run.py:852] *****************************************
<frozen runpy>:128: RuntimeWarning: 'basicsr.train' found in sys.modules after import of package 'basicsr', but prior to execution of 'basicsr.train'; this may result in unpredictable behaviour
<frozen runpy>:128: RuntimeWarning: 'basicsr.train' found in sys.modules after import of package 'basicsr', but prior to execution of 'basicsr.train'; this may result in unpredictable behaviour
pretrain_network path will be ignored during resuming.
Set pret

In [5]:
# 检查试运行结果
log_dir = os.path.join(exp_dir)
logs = sorted([f for f in os.listdir(log_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(log_dir, logs[-1])
    with open(latest_log, 'r') as f:
        lines = f.readlines()
    # 打印最后 10 行
    print(f'日志: {logs[-1]}')
    print('='*60)
    for line in lines[-10:]:
        print(line.rstrip())
    
    # 检查是否有报错
    errors = [l for l in lines if 'Error' in l or 'Traceback' in l]
    if errors:
        print(f'\n❌ 发现 {len(errors)} 个错误！')
    else:
        # 提取 ETA
        eta_lines = [l for l in lines if 'eta' in l]
        if eta_lines:
            print(f'\n✅ 试运行成功！最后的 ETA: {eta_lines[-1].strip()}')
else:
    print('❌ 未找到日志文件')

日志: train_finetune_SwinIR_SRx4_SEM_20260709_162657.log
2026-07-09 16:28:15,398 INFO: Start training from epoch: 2, iter: 10000
2026-07-09 16:29:35,523 INFO: [finet..][epoch:  2, iter:  10,050, lr:(1.000e-04,)] [eta: 0:05:04, time (data): 1.602 (0.582)] l_pix: 2.6386e-02
2026-07-09 16:30:30,878 INFO: [finet..][epoch:  2, iter:  10,100, lr:(1.000e-04,)] [eta: 0:02:36, time (data): 1.355 (0.293)] l_pix: 3.1597e-02
2026-07-09 16:31:23,811 INFO: [finet..][epoch:  2, iter:  10,150, lr:(1.000e-04,)] [eta: 0:01:08, time (data): 1.256 (0.196)] l_pix: 4.8078e-02
2026-07-09 16:32:17,723 INFO: [finet..][epoch:  2, iter:  10,200, lr:(1.000e-04,)] [eta: -1 day, 23:59:59, time (data): 1.212 (0.148)] l_pix: 3.8558e-02
2026-07-09 16:35:35,907 INFO: End of training. Time consumed: 0:07:20
2026-07-09 16:35:35,907 INFO: Save the latest model.
2026-07-09 16:40:53,315 INFO: Validation LVSEM_eval
	 # psnr: 26.5756	Best: 26.5756 @ 10208 iter


✅ 试运行成功！最后的 ETA: 2026-07-09 16:32:17,723 INFO: [finet..][epoch:  2

## 2. 正式训练

试运行确认无误后，运行此 cell。训练 70k iter，预计 10-12 小时（2×T4）。

**注意**：Kaggle Notebook 有 12 小时限制。如果超时，`auto_resume: true` 会在下次运行时自动恢复。

In [ ]:
# 正式训练 — DDP（torchrun）+ auto_resume 断点续训
# Kaggle 12h 限制，auto_resume 确保超时后下次运行自动恢复
print('开始正式训练 (70k iter, DDP 2xT4)...')
print('='*50)
!torchrun --nproc_per_node=2 --master_port=4321 -m basicsr.train \
  -opt options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml \
  --launcher pytorch --auto_resume

## 3. 结果分析

In [ ]:
# 提取所有验证结果
logs = sorted([f for f in os.listdir(exp_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(exp_dir, logs[-1])
    with open(latest_log, 'r') as f:
        content = f.read()
    
    # 提取验证 PSNR
    import re
    val_matches = re.findall(r'psnr:\s+([\d.]+)\s+Best:\s+([\d.]+)\s+@\s+(\d+)', content)
    
    if val_matches:
        print('验证结果:')
        print(f'{"Iter":<10} {"PSNR":<10} {"Best PSNR":<12}')
        print('-' * 35)
        best_psnr = 0
        for psnr_val, best_val, iter_val in val_matches:
            marker = ' ←' if float(psnr_val) == float(best_val) else ''
            print(f'{iter_val:<10} {psnr_val:<10} {best_val:<12}{marker}')
    
    # 最后几行
    lines = content.strip().split('\n')
    print(f'\n日志最后 5 行:')
    for line in lines[-5:]:
        print(line)
else:
    print('未找到日志')

## 4. 保存结果到 output

训练完成后，output 目录中的文件会被保存为 Kaggle Dataset，可以下载。

In [ ]:
# /kaggle/working 整个目录在 Save and Run All 后会自动保存为 output
# 不需要手动复制，直接查看训练产物即可

print('训练产物位置:')
print(f'  模型权重: {models_dir}')
print(f'  训练状态: {states_dir}')

# 列出模型文件
print(f'\n模型文件:')
for f in sorted(os.listdir(models_dir)):
    size = os.path.getsize(os.path.join(models_dir, f)) / 1e6
    print(f'  {f} ({size:.1f}MB)')

# 列出训练状态文件
print(f'\n训练状态:')
for f in sorted(os.listdir(states_dir)):
    size = os.path.getsize(os.path.join(states_dir, f)) / 1e6
    print(f'  {f} ({size:.1f}MB)')

# 列出日志
print(f'\n日志文件:')
for f in sorted(os.listdir(exp_dir)):
    if f.endswith('.log'):
        size = os.path.getsize(os.path.join(exp_dir, f)) / 1e6
        print(f'  {f} ({size:.1f}MB)')


In [ ]:
# Kaggle Save and Run All 会自动保存 /kaggle/working 下所有内容
# 右侧面板 → Data → 下载即可

working_size = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk('/kaggle/working')
    for f in files
)
print(f'working 目录总大小: {working_size/1e9:.2f} GB')
print(f'训练完成！Save and Run All 后，右侧面板 → Data → 下载。')